In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 28 — EXPLICABILIDADE CLÍNICA: DA TEORIA À PRÁTICA MÉDICA

> "Uma explicação só é boa se for compreendida. Mostrar um gráfico SHAP a um médico ocupado é como mostrar uma partitura a quem quer ouvir a música. Você precisa tocar a melodia."
> — Uma Designer de Interação Humano-Computador

Eu tinha as ferramentas: SHAP e LIME dissecavam qualquer previsão. Mas a verdade, em sua forma bruta, nem sempre é útil. Imaginei apresentar o sistema aos médicos. Um paciente chega, o modelo prevê "Maligno" com 85%. O médico, cético, pergunta "Por quê?". Eu mostro um gráfico de força do SHAP, com barras vermelhas e azuis. Ele olha, confuso — não tem tempo de decifrar uma nova visualização no meio da consulta. Ele precisa de uma resposta na *sua* linguagem.

A última milha da interpretabilidade não era técnica: era de tradução. Eu precisava pegar a saída rica das ferramentas de XAI e traduzi-la numa narrativa simples, concisa e clinicamente relevante. Não mostrar os cálculos — contar a história que eles revelam.

## Traduzindo SHAP Para a Linguagem Médica

Explicabilidade clínica é projetar a saída do sistema para ser diretamente útil a um profissional de saúde. Cinco princípios:

**Relevância** — destacar os 3 fatores principais, não os 30.

**Linguagem natural** — "a principal razão é a alta irregularidade do contorno", não "o valor SHAP foi −0,45".

**Contextualização** — mostrar o valor do paciente ao lado da referência.

**Confiança e incerteza** — comunicar se a previsão é de 95% ou de 55%.

Vamos construir uma função que transforma os valores SHAP de uma previsão nessa narrativa. Ela é o coração da interface do OncoClassify 2.0.

#### Código 28.1: Gerando Explicações Clínicas em Linguagem Natural


In [ ]:
import numpy as np, pandas as pd, shap
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from oncoclassify_utils import X_train, y_train, FEATURES_MORFOLOGICAS
np.random.seed(42)

def info_mutua(X, y): return mutual_info_classif(X, y, random_state=42)
modelo = Pipeline([("selecao", SelectKBest(info_mutua, k=25)),
                   ("escala", StandardScaler()),
                   ("svm", SVC(kernel="rbf", C=10, gamma=0.01,
                               probability=True, random_state=42))]).fit(X_train, y_train)

# explainer SHAP (como no Cap. 19) para gerar as contribuicoes de cada paciente
def prob_maligno(dados):
    return modelo.predict_proba(pd.DataFrame(dados, columns=FEATURES_MORFOLOGICAS))[:, 0]
explainer = shap.KernelExplainer(prob_maligno, shap.kmeans(X_train.values, 15))

# --- a funcao que traduz SHAP em linguagem clinica ---
def explicacao_clinica(paciente, shap_values, prob):
    diagnostico = "Maligno" if prob >= 0.5 else "Benigno"
    confianca   = "alta" if (prob > 0.85 or prob < 0.15) else "moderada"
    # fatores A FAVOR do diagnostico = SHAP na direcao prevista
    favor = (shap_values[shap_values > 0].sort_values(ascending=False)
             if diagnostico == "Maligno"
             else shap_values[shap_values < 0].sort_values())
    linhas = ["**Relatorio OncoClassify v2.0**",
              f"- Previsao: {diagnostico}",
              f"- Confianca: {confianca} ({prob:.0%} de probabilidade de malignidade)",
              f"- Principais fatores ({diagnostico.lower()}):"]
    for feature in favor.index[:3]:
        linhas.append(f"    - {feature} = {paciente[feature]:.3f}")
    return "\n".join(linhas)

# escolhe dois pacientes do TREINO (o cofre X_test segue lacrado ate o Cap. 24):
# um maligno confiante e um benigno confiante
proba = modelo.predict_proba(X_train)[:, 0]
idx_A = [i for i in range(len(X_train)) if y_train.values[i] == 0 and proba[i] > 0.9][0]
idx_B = [i for i in range(len(X_train)) if y_train.values[i] == 1 and proba[i] < 0.1][0]

for idx in (idx_A, idx_B):
    x = X_train.iloc[idx]
    sv = pd.Series(explainer.shap_values(x.values.reshape(1, -1), nsamples=100)[0],
                   index=FEATURES_MORFOLOGICAS)
    print(explicacao_clinica(x, sv, proba[idx]))
    print()

**Relatorio OncoClassify v2.0**
- Previsao: Maligno
- Confianca: alta (99% de probabilidade de malignidade)
- Principais fatores (maligno):
    - worst concave points = 0.239
    - worst area = 980.900
    - mean concave points = 0.098

**Relatorio OncoClassify v2.0**
- Previsao: Benigno
- Confianca: alta (0% de probabilidade de malignidade)
- Principais fatores (benigno):
    - mean compactness = 0.197
    - worst perimeter = 68.620
    - compactness error = 0.066


É exatamente o que um médico gostaria de ver no prontuário. A saída é **concisa** (só o essencial), em **linguagem natural**, **contextualizada** (mostra o valor do paciente) e **consciente da incerteza** (rotula a confiança). Para o Paciente A, o modelo aponta malignidade pela **irregularidade e tamanho** do núcleo (pontos côncavos 0,239, área 980,9); para o Paciente B, benignidade por um núcleo **pequeno e regular** (perímetro 68,6).  É a ponte final entre a complexidade do modelo e a clareza de que o usuário precisa — o design da interação humano-IA em ação.

> **📌 CHECKLIST DO CAPÍTULO 28**
>
> ✅ Entende a diferença entre explicabilidade **técnica** e **clínica**.
>
> ✅ Conhece os princípios: relevância, linguagem natural, contextualização e incerteza.
>
> ✅ Executou o Código 28.1 e traduziu os valores SHAP numa narrativa útil para dois pacientes reais.
>
> ✅ Reconhece que a interpretabilidade não termina no algoritmo de XAI, mas na **interface** que apresenta a explicação.
>
> **⚠️ CRÍTICO:** A melhor explicação do mundo é inútil se o usuário não puder consumi-la. Investir em design de UI/UX para IA é tão importante quanto investir no modelo.

Com a função que gerava explicações em texto, senti que havia fechado o ciclo da responsabilidade. Meu sistema não apenas previa: explicava, numa forma que respeitava o tempo e a experiência do médico. Eu havia construído não só uma arquitetura da verdade, mas uma arquitetura da confiança.

Minha jornada técnica e ética estava completa. Restava a parte mais humana da história: refletir sobre o caminho, comparar o resultado final com o ponto de partida e, enfim, responder à carta de Helena — não com um pedido de desculpas, mas com uma demonstração de progresso.

**PARTE IX — FECHAMENTO E REFLEXÃO**
